# Workshop 1 · Gold KPI Model for Power BI

![Gold model contract](../../assets/images/w1_gold_model_contract.png)

In this workshop you build the complete RetailHub Gold model used by the optional Day1_03 checkpoint and the Day 2 Power BI handoff. This is the place where the full Kimball-style model is created; the earlier demos only prepared the concepts.


## Business Scenario

RetailHub has multiple Power BI reports with inconsistent revenue, margin and order counts. The data exists in Databricks Silver, but Power BI developers need a governed Gold contract before building a certified semantic model.

In this workshop you play the role of a Power BI developer working with the Databricks platform. Your job is not only to create tables; your job is to make report-ready decisions explicit: grain, keys, unknown handling, KPI filters and relationship readiness.

## Target Contract

By the end, the Gold schema must contain these durable objects:

| Object | Grain | Purpose |
|---|---|---|
| `gold.dim_date` | one row per calendar date | date slicers, calendar attributes, relationship target |
| `gold.dim_customer` | one row per customer | customer geography and segment slicers |
| `gold.dim_product` | one row per product | product category and active-product slicers |
| `gold.fact_sales` | one row per order line | governed measures and relationship source |
| `gold.fact_sales_dashboard` | one row per order line | wide table for fast Power BI report prototyping |

Expected observation: `fact_sales` and `fact_sales_dashboard` keep the same line grain. The dashboard table is wider, not more aggregated.


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `silver.order_lines`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before running SQL cells.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

This notebook requires the four Silver source tables created by `data/generate_training_dataset.ipynb`.


In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run data/generate_training_dataset.ipynb first.")
print("[OK] Silver source tables are available.")


### Workshop Validation Helpers

These helper functions keep the task-level assertions short and readable. They are used only for validation, not for the main SQL implementation.


In [ ]:
# -- Validation helpers --
def _table_exists(full_name):
    return spark.catalog.tableExists(full_name)


def _require_table(full_name):
    assert _table_exists(full_name), f"Missing object: {full_name}"


def _columns(full_name):
    return set(spark.table(full_name).columns)


def _require_columns(full_name, required_columns):
    missing = sorted(set(required_columns) - _columns(full_name))
    assert not missing, f"{full_name} missing columns: {missing}"


def _scalar(query):
    row = spark.sql(query).first()
    return row[0] if row is not None else None


def _first(query):
    row = spark.sql(query).first()
    assert row is not None, "Query returned no rows"
    return row


def _print_ok(message):
    print(message)


## Workshop Rubric

A correct solution must satisfy these checks:

- every standalone SQL step is a `%sql` cell,
- dimensions have unique keys,
- `gold.fact_sales` has one row per `line_id`,
- `gold.fact_sales_dashboard` keeps one row per `line_id`,
- date, customer and product relationships can be built as many-to-one relationships in Power BI,
- KPI rules are explicit: revenue and margin count completed rows only,
- unknown dimension attributes are visible through controlled labels, not silent row loss,
- model-change comparison explains when a different filter changes the KPI and when a different table shape should preserve the KPI.

Power BI developer lens: by the end of this workshop you should be able to defend why Power BI should use these Gold objects instead of importing raw Silver tables.

## Source Discovery Before Modeling

Business context: before creating a semantic model, prove what the source tables contain. Most BI defects start with a wrong assumption about grain or status filters.

Why Power BI cares: a line-grain table, an order-grain table and a dashboard aggregate produce different DAX behavior even when column names look similar.

Required output: row counts, distinct keys, date range and status distribution from Silver. Use the next query as your starting evidence.

In [ ]:
%sql
SELECT 'customers' AS source_table, COUNT(*) AS rows FROM silver.customers
UNION ALL
SELECT 'products', COUNT(*) FROM silver.products
UNION ALL
SELECT 'sales_orders', COUNT(*) FROM silver.sales_orders
UNION ALL
SELECT 'order_lines', COUNT(*) FROM silver.order_lines
ORDER BY source_table


In [ ]:
%sql
SELECT
  COUNT(*) AS line_rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS avg_lines_per_order,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM silver.order_lines


## Task 1: Create Gold Schema

Business context: Gold is the certified BI contract. Power BI developers should not point governed reports directly at raw or partially modeled source tables.

Why Power BI cares: a stable schema name makes handoff, refresh configuration and permissions predictable.

Required output: schema `gold` exists.

Implementation hints:

- use `CREATE SCHEMA IF NOT EXISTS`,
- do not create another schema name,
- keep this step idempotent so the workshop can be rerun.

Acceptance criteria: `SHOW SCHEMAS LIKE 'gold'` returns one row.

Common mistake: creating personal schema names breaks downstream notebooks and handoff checklists.

**Guidance — Task 1**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create the Gold schema if it does not exist.
-- Acceptance: SHOW SCHEMAS LIKE 'gold' returns one row.


In [ ]:
# -- Validation --
schema_count = spark.sql(f"SHOW SCHEMAS IN {CATALOG} LIKE 'gold'").count()
assert schema_count == 1, "Task 1 failed: schema 'gold' was not found in the participant catalog"
_print_ok("Task 1 OK: Gold schema exists and is ready for BI objects.")


In [ ]:
%sql
SHOW SCHEMAS LIKE 'gold'


## Task 2: Build `gold.dim_date`

Business context: Power BI semantic models need a consistent date table for slicers, trend charts and time intelligence.

Why Power BI cares: a proper date dimension prevents report authors from building separate calendar logic in every `.pbix` file.

Required output: `gold.dim_date` from `2024-01-01` to `2026-12-31`.

Required columns: `date_key`, `year`, `quarter`, `month`, `month_name`, `year_month`, `day_of_month`, `day_of_week`, `day_name`, `is_weekend`.

Implementation hints:

- generate a date sequence with `explode(sequence(...))`,
- use `date_key` as the relationship target,
- derive labels such as `year_month` in Gold so Power BI gets consistent formatting.

Acceptance criteria: one row per date, no duplicate `date_key`, min/max dates match the required range.

Common mistake: using string dates as relationship keys or omitting `year_month` needed by BI visuals.

**Guidance — Task 2**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace gold.dim_date.
-- Hint: use explode(sequence(to_date('2024-01-01'), to_date('2026-12-31'), interval 1 day)).


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT date_key) AS distinct_dates,
  COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_date_keys,
  MIN(date_key) AS min_date,
  MAX(date_key) AS max_date
FROM gold.dim_date


In [ ]:
# -- Validation --
table = f"{GOLD}.dim_date"
_require_table(table)
_require_columns(table, ["date_key", "year", "quarter", "month", "month_name", "year_month", "day_of_month", "day_of_week", "day_name", "is_weekend"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT date_key) AS duplicate_keys, MIN(date_key) AS min_date, MAX(date_key) AS max_date
FROM {GOLD}.dim_date
""")
assert row["rows"] > 0, "Task 2 failed: dim_date is empty"
assert row["duplicate_keys"] == 0, f"Task 2 failed: duplicate date_key rows = {row['duplicate_keys']}"
assert str(row["min_date"]) == "2024-01-01", f"Task 2 failed: min date is {row['min_date']}"
assert str(row["max_date"]) == "2026-12-31", f"Task 2 failed: max date is {row['max_date']}"
_print_ok(f"Task 2 OK: dim_date has {row['rows']} unique calendar dates from 2024-01-01 to 2026-12-31.")


## Task 3: Build `gold.dim_customer`

Business context: customer attributes drive slicers, segmentation and geography cuts in Power BI.

Why Power BI cares: dimension columns must be clean, readable and unique on the relationship key.

Required output: `gold.dim_customer` with one row per customer.

Required columns: `customer_id`, `customer_name`, `city`, `customer_region`, `country`, `segment`, `created_date`.

Implementation hints:

- source table: `silver.customers`,
- rename `region AS customer_region` so report fields are business-friendly,
- do not aggregate customers,
- keep the relationship key as `customer_id`.

Acceptance criteria: one row per `customer_id`.

Common mistake: leaving a generic `region` column when the report also has product or source regions.

**Guidance — Task 3**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace gold.dim_customer from silver.customers.
-- Required rename: region AS customer_region.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT customer_id) AS distinct_customers,
  COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customer_keys
FROM gold.dim_customer


In [ ]:
# -- Validation --
table = f"{GOLD}.dim_customer"
_require_table(table)
_require_columns(table, ["customer_id", "customer_name", "city", "customer_region", "country", "segment", "created_date"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_keys, SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM {GOLD}.dim_customer
""")
assert row["rows"] > 0, "Task 3 failed: dim_customer is empty"
assert row["duplicate_keys"] == 0, f"Task 3 failed: duplicate customer_id rows = {row['duplicate_keys']}"
assert row["null_keys"] == 0, f"Task 3 failed: null customer_id rows = {row['null_keys']}"
_print_ok(f"Task 3 OK: dim_customer has {row['rows']} unique customer keys and BI-ready attributes.")


## Task 4: Build `gold.dim_product`

Business context: product dimensions define category and subcategory slicers used in KPI dashboards.

Why Power BI cares: product attributes should be conformed once in Gold, not recreated through Power Query steps.

Required output: `gold.dim_product` with one row per product.

Required columns: `product_id`, `product_name`, `category`, `subcategory`, `standard_unit_cost`, `list_unit_price`, `is_active`.

Implementation hints:

- source table: `silver.products`,
- rename `unit_cost AS standard_unit_cost`,
- rename `unit_price AS list_unit_price`,
- preserve `is_active` for slicers and data-quality review.

Acceptance criteria: one row per `product_id`.

Common mistake: using transactional unit price as list price in dimension logic.

**Guidance — Task 4**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace gold.dim_product from silver.products.
-- Required renames: unit_cost AS standard_unit_cost, unit_price AS list_unit_price.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT product_id) AS distinct_products,
  COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_product_keys
FROM gold.dim_product


In [ ]:
# -- Validation --
table = f"{GOLD}.dim_product"
_require_table(table)
_require_columns(table, ["product_id", "product_name", "category", "subcategory", "standard_unit_cost", "list_unit_price", "is_active"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_keys, SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM {GOLD}.dim_product
""")
assert row["rows"] > 0, "Task 4 failed: dim_product is empty"
assert row["duplicate_keys"] == 0, f"Task 4 failed: duplicate product_id rows = {row['duplicate_keys']}"
assert row["null_keys"] == 0, f"Task 4 failed: null product_id rows = {row['null_keys']}"
_print_ok(f"Task 4 OK: dim_product has {row['rows']} unique product keys and conformed attributes.")


## Task 5: Build `gold.fact_sales`

Business context: the fact table is the governed line-grain source for reusable sales measures.

Why Power BI cares: if the fact grain is unclear, DAX measures can double-count revenue or orders.

Required output: `gold.fact_sales` at one row per `line_id`.

Required keys: `line_id`, `order_id`, `order_date`, `customer_id`, `product_id`.

Required measures: `quantity`, `unit_price`, `unit_cost`, `discount_pct`, `line_revenue`, `line_cost`, `line_margin`.

Required flags: `is_completed`, `is_returned`, `is_cancelled`.

Implementation hints:

- source table: `silver.order_lines`,
- keep all rows at line grain,
- define flags from `status`,
- keep status text for troubleshooting and flags for BI-friendly measures.

Acceptance criteria: duplicate `line_id` count is zero.

Common mistake: aggregating to order level too early and losing drill-through detail.

**Guidance — Task 5**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace gold.fact_sales.
-- Acceptance: duplicate_line_ids = 0 in the next check.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_line_ids,
  COUNT(DISTINCT order_id) AS distinct_orders
FROM gold.fact_sales


In [ ]:
# -- Validation --
table = f"{GOLD}.fact_sales"
_require_table(table)
_require_columns(table, ["line_id", "order_id", "order_date", "customer_id", "product_id", "channel", "status", "quantity", "unit_price", "unit_cost", "discount_pct", "line_revenue", "line_cost", "line_margin", "is_completed", "is_returned", "is_cancelled"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_line_ids,
       SUM(CASE WHEN line_id IS NULL OR order_id IS NULL OR order_date IS NULL OR customer_id IS NULL OR product_id IS NULL THEN 1 ELSE 0 END) AS null_required_keys
FROM {GOLD}.fact_sales
""")
assert row["rows"] > 0, "Task 5 failed: fact_sales is empty"
assert row["duplicate_line_ids"] == 0, f"Task 5 failed: duplicate line_id rows = {row['duplicate_line_ids']}"
assert row["null_required_keys"] == 0, f"Task 5 failed: rows with null required keys = {row['null_required_keys']}"
_print_ok(f"Task 5 OK: fact_sales has {row['rows']} line-grain rows with required keys and measures.")


## Task 6: Build `gold.fact_sales_dashboard`

Business context: this is a wide, report-friendly table for quick dashboard prototypes and simple Import models.

Why Power BI cares: a wide table can speed up simple reporting, but it must preserve the same line grain as the governed fact table.

Required behavior:

- keep one row per `line_id`,
- include date, customer and product attributes used by slicers,
- use `COALESCE` for missing dimension attributes so blank slicers become visible labels,
- do not aggregate rows in this table.

Implementation hints:

- start from `gold.fact_sales`,
- left join dimensions to avoid dropping fact rows,
- use controlled labels such as `Unknown Customer`, `Unknown Product`, `Unknown`,
- keep measure fields from the fact table unchanged.

Acceptance criteria: duplicate line count is zero and row count equals `gold.fact_sales`.

Common mistake: using inner joins, which can silently remove facts when a dimension is incomplete.

**Guidance — Task 6**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace gold.fact_sales_dashboard.
-- Acceptance: duplicate_lines = 0 and rows = rows in gold.fact_sales.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_lines
FROM gold.fact_sales_dashboard


In [ ]:
# -- Validation --
table = f"{GOLD}.fact_sales_dashboard"
_require_table(table)
_require_columns(table, ["line_id", "order_id", "order_date", "year_month", "customer_id", "customer_name", "customer_region", "product_id", "product_name", "category", "subcategory", "channel", "status", "line_revenue", "line_margin", "is_completed", "is_returned", "is_cancelled"])
row = _first(f"""
SELECT (SELECT COUNT(*) FROM {GOLD}.fact_sales) AS fact_rows, COUNT(*) AS dashboard_rows, COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_lines
FROM {GOLD}.fact_sales_dashboard
""")
assert row["dashboard_rows"] > 0, "Task 6 failed: fact_sales_dashboard is empty"
assert row["duplicate_lines"] == 0, f"Task 6 failed: duplicate dashboard line rows = {row['duplicate_lines']}"
assert row["dashboard_rows"] == row["fact_rows"], f"Task 6 failed: dashboard rows {row['dashboard_rows']} do not match fact rows {row['fact_rows']}"
_print_ok(f"Task 6 OK: fact_sales_dashboard preserves line grain and matches {row['fact_rows']} fact rows.")


## Task 7: Relationship Readiness

Business context: before building a Power BI semantic model, prove the relationship contract.

Why Power BI cares: many-to-one relationships require unique dimension keys and matching fact keys. Orphans create blank slicer values or misleading totals.

Required output: one row per relationship with `orphan_rows`.

Required relationships:

- `fact_sales.order_date -> dim_date.date_key`,
- `fact_sales.customer_id -> dim_customer.customer_id`,
- `fact_sales.product_id -> dim_product.product_id`.

Implementation hints:

- use `LEFT ANTI JOIN` to find missing dimension rows,
- return all three checks in one result with `UNION ALL`,
- do not “fix” orphans in the check query; report them.

Acceptance criteria: each relationship returns zero orphan rows, or the source-quality reason is documented.

Common mistake: checking only row counts and skipping relationship validity.

**Guidance — Task 7**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: return one row per relationship with orphan_rows.
-- Required relationships:
-- fact_sales.order_date -> dim_date.date_key
-- fact_sales.customer_id -> dim_customer.customer_id
-- fact_sales.product_id -> dim_product.product_id


In [ ]:
# -- Validation --
row = _first(f"""
WITH checks AS (
  SELECT COUNT(*) AS orphan_rows FROM {GOLD}.fact_sales f LEFT ANTI JOIN {GOLD}.dim_date d ON f.order_date = d.date_key
  UNION ALL SELECT COUNT(*) FROM {GOLD}.fact_sales f LEFT ANTI JOIN {GOLD}.dim_customer c ON f.customer_id = c.customer_id
  UNION ALL SELECT COUNT(*) FROM {GOLD}.fact_sales f LEFT ANTI JOIN {GOLD}.dim_product p ON f.product_id = p.product_id
)
SELECT SUM(orphan_rows) AS total_orphan_rows FROM checks
""")
assert row["total_orphan_rows"] == 0, f"Task 7 failed: relationship orphan rows = {row['total_orphan_rows']}"
_print_ok("Task 7 OK: fact_sales has zero date/customer/product orphan rows.")


## Task 8: KPI Smoke Test

Business context: KPI definitions must be stable before report authors create DAX measures.

Why Power BI cares: if completed revenue is not defined in Gold, every report author may implement a different filter.

Compute the baseline KPI set from `gold.fact_sales_dashboard`.

Rules:

- Revenue: sum `line_revenue` for completed rows only.
- Gross Margin: sum `line_margin` for completed rows only.
- Completed Orders: distinct completed `order_id`.
- Margin Rate %: Gross Margin divided by Revenue, protected with `NULLIF`.

Implementation hints:

- use `CASE WHEN is_completed THEN ... END`,
- protect division with `NULLIF`,
- use `COUNT(DISTINCT order_id)` for order count, not line count.

Acceptance criteria: revenue, margin and completed orders are non-null and margin rate is not a divide-by-zero error.

Common mistake: counting order lines as orders.

**Guidance — Task 8**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: compute revenue, gross_margin, completed_orders and margin_rate_pct from gold.fact_sales_dashboard.


In [ ]:
# -- Validation --
row = _first(f"""
SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
       ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
       COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
       ROUND(100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END) / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0), 2) AS margin_rate_pct
FROM {GOLD}.fact_sales_dashboard
""")
assert row["revenue"] is not None and row["revenue"] > 0, "Task 8 failed: completed revenue must be positive"
assert row["gross_margin"] is not None, "Task 8 failed: gross margin is null"
assert row["completed_orders"] > 0, "Task 8 failed: completed order count must be positive"
assert row["margin_rate_pct"] is not None, "Task 8 failed: margin rate is null; protect division with NULLIF"
_print_ok(f"Task 8 OK: KPI smoke test passed for {row['completed_orders']} completed orders.")


## Task 9: Model Change Impact Check

Business context: Power BI developers must understand which changes are business-definition changes and which are only model-shape changes.

You will compare two things:

1. KPI filter logic: completed-only revenue vs a broader completed-or-returned scenario.
2. Query shape: star schema query vs wide dashboard-table query.

Why Power BI cares:

- Changing the KPI filter should change the number because the business definition changed.
- Changing from star schema to wide table should not change the number if both queries implement the same KPI definition.

Required output:

- `w1_status_filter_comparison` TEMP VIEW with `scenario_name`, `revenue`, `order_count`,
- `w1_model_pattern_comparison` TEMP VIEW with `pattern_name`, `revenue`, `line_rows`.

Implementation hints:

- use `gold.fact_sales_dashboard` for the filter comparison,
- use `gold.fact_sales` plus dimensions for the star schema row,
- use `gold.fact_sales_dashboard` for the wide dashboard row,
- keep both model-pattern rows on completed sales only.

Acceptance criteria:

- status-filter delta is explainable as a business-definition change,
- star-vs-wide revenue delta is zero,
- no durable Gold object is changed.

Common mistake: treating a star schema and a wide table as different KPI definitions instead of different consumption shapes.

**Guidance — Task 9**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create TEMP VIEW w1_status_filter_comparison.
-- Required columns:
--   scenario_name
--   revenue
--   order_count
-- Required rows:
--   completed_only
--   completed_or_returned
-- Hint: use gold.fact_sales_dashboard and explicit status flags.

In [ ]:
%sql
-- TODO: create TEMP VIEW w1_model_pattern_comparison.
-- Required columns:
--   pattern_name
--   revenue
--   line_rows
-- Required rows:
--   star_schema
--   wide_dashboard
-- Rule: both rows should use completed sales only.

In [ ]:
%sql
-- Check after your TODO views exist.
SELECT
  'status_filter_delta' AS check_name,
  ROUND(
    MAX(CASE WHEN scenario_name = 'completed_or_returned' THEN revenue END)
    - MAX(CASE WHEN scenario_name = 'completed_only' THEN revenue END),
    2
  ) AS observed_delta
FROM w1_status_filter_comparison
UNION ALL
SELECT
  'star_vs_wide_revenue_delta',
  ROUND(
    MAX(CASE WHEN pattern_name = 'star_schema' THEN revenue END)
    - MAX(CASE WHEN pattern_name = 'wide_dashboard' THEN revenue END),
    2
  )
FROM w1_model_pattern_comparison

In [ ]:
# -- Validation --
status_row = _first("""
SELECT ROUND(MAX(CASE WHEN scenario_name = 'completed_or_returned' THEN revenue END) - MAX(CASE WHEN scenario_name = 'completed_only' THEN revenue END), 2) AS status_filter_delta,
       COUNT(DISTINCT scenario_name) AS scenarios
FROM w1_status_filter_comparison
""")
pattern_row = _first("""
SELECT ROUND(MAX(CASE WHEN pattern_name = 'star_schema' THEN revenue END) - MAX(CASE WHEN pattern_name = 'wide_dashboard' THEN revenue END), 2) AS star_vs_wide_delta,
       COUNT(DISTINCT pattern_name) AS patterns
FROM w1_model_pattern_comparison
""")
assert status_row["scenarios"] == 2, "Task 9 failed: status comparison must contain two scenarios"
assert pattern_row["patterns"] == 2, "Task 9 failed: model pattern comparison must contain two patterns"
assert abs(float(pattern_row["star_vs_wide_delta"])) < 0.01, f"Task 9 failed: star vs wide revenue delta = {pattern_row['star_vs_wide_delta']}"
_print_ok(f"Task 9 OK: status filter delta is {status_row['status_filter_delta']} and star-vs-wide revenue reconciles.")


Expected observation: the status filter delta should explain a business definition change. The star-vs-wide revenue delta should be zero when both queries implement the same KPI rule.

## Task 10: Object Inventory and Handoff

Business context: a Power BI handoff needs an object inventory before anyone opens Power BI Desktop.

Why Power BI cares: report authors need to know which tables are official, their expected row counts and which object is best for star-schema modeling vs quick dashboard prototyping.

Required output: row counts for all five Gold objects.

Implementation hints:

- return object names exactly as `gold.dim_date`, `gold.dim_customer`, `gold.dim_product`, `gold.fact_sales`, `gold.fact_sales_dashboard`,
- include one row per object,
- use this output as the transition into the Day 2 semantic-model module.

Acceptance criteria: all five objects exist and return rows.

**Guidance — Task 10**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: return row counts for all five Gold objects.


In [ ]:
# -- Validation --
objects = [f"{GOLD}.dim_date", f"{GOLD}.dim_customer", f"{GOLD}.dim_product", f"{GOLD}.fact_sales", f"{GOLD}.fact_sales_dashboard"]
counts = []
for object_name in objects:
    _require_table(object_name)
    row_count = spark.table(object_name).count()
    assert row_count > 0, f"Task 10 failed: {object_name} has no rows"
    counts.append((object_name, row_count))
_print_ok("Task 10 OK: Gold handoff inventory is complete: " + ", ".join(f"{name}={rows}" for name, rows in counts))


## Completion Checklist

Before moving to the optional checkpoint or Day 2, confirm:

- all five target objects exist,
- dimensions have unique keys,
- fact and dashboard tables are line-grain,
- dashboard table includes date, customer, product, channel, status and measure columns,
- KPI smoke test returns non-null business values,
- model-change task explains filter impact and star-vs-wide consistency,
- relationship readiness check returns zero orphan rows or a documented source-quality reason.
